# 18. H18: 딥러닝 모델은 트리·선형 모델과 다르게 틀려서 섞기 점수를 올린다

## 배경
- 14·16에서 XGB + LR (+RF) 섞기가 LB로 넘어감 (0.3206 → 0.3247)
- 교훈: 피처 추가로 오른 CV는 LB로 안 넘어가고, 과적합을 줄이거나 섞어서 오른 CV는 넘어감
- 딥러닝 단독은 XGB보다 낮을 가능성이 높음 → **목표는 단독 점수가 아니라 섞기 재료**

## 가설
- H18a: MLP(변이 여부 + stats)는 LR과 XGB 사이 성격이라 섞으면 OOF가 오른다
- H18b: 변이 유전자만 토큰으로 넣는 트랜스포머는 유전자끼리 같이 변이되는 패턴을 직접 봐서, 더 다른 방식으로 틀린다

## 설계
- 피처: 10번 설정 그대로 (정규화 후 any + stats), 핫스팟 제외 (11에서 LB 효과 없음)
- 증강: 학습 중 일부 환자에 **학습 fold 유전자 빈도**로 가짜 변이 추가 + 진짜 변이 10% 삭제
  - test 환자당 변이 수가 train의 약 2배라, 변이 수 변화에 덜 흔들리게 하려는 목적
- 누수 방지: stats 표준화, 클래스 가중치, 증강 빈도, 조기 종료 모두 **fit 안의 학습 fold로만** 계산 (`src/dl_models.py`)
- 같은 `run_cv`라 fold가 기존 실험과 같음 → OOF끼리 바로 섞을 수 있음

## 채택 기준 (미리 정함)
- 16 최종 섞기에 딥러닝을 더했을 때, **교차 적합 OOF가 5 fold 중 4개 이상에서 오르면** 12일 제출 1회에 사용
- 아니면 종료하고 발표에 "시도했지만 트리 섞기가 나았다"로 기록

In [1]:
import sys; sys.path.append('..')
import time, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.metrics import f1_score
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, get_folds, run_cv, macro_f1, save_submission
from src.dl_models import MLPClassifierTorch, GeneSetTransformer, get_device

seed_everything()
print('device:', get_device(), '| torch', torch.__version__)

train, test, submission = load_data()   # 정규화 + dedupe=True (10번과 동일)
genes = get_gene_cols(train)
y, le = encode_target(train)
FOLDS = get_folds(y)                   # 16과 같은 fold
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

X      = pd.concat([F_tr['any'], F_tr['stats']], axis=1)
X_test = pd.concat([F_te['any'], F_te['stats']], axis=1)
N_ANY  = F_tr['any'].shape[1]          # 앞 N_ANY개 열 = 유전자 0/1, 나머지 = stats
N_CLS  = len(le.classes_)
print(X.shape, X_test.shape, 'n_any', N_ANY, 'classes', N_CLS)

CACHE = Path('../submissions/cache17'); CACHE.mkdir(parents=True, exist_ok=True)

device: cuda | torch 2.15.0+cu132
(6201, 4393) (2546, 4393) n_any 4384 classes 26


## 1) MLP
- 2층 (512 → 128), 드롭아웃 0.5, 입력 드롭아웃 0.1, 클래스 가중치, 조기 종료 (학습 fold 안 10%)
- 4060 Ti 기준 fold당 1분 안팎, 맥북 CPU는 몇 분

In [2]:
t = time.time()
make_mlp = lambda: MLPClassifierTorch(N_ANY, n_classes=N_CLS, seed=SEED)
r_mlp = run_cv(make_mlp, X, y, X_test, folds=FOLDS)
np.savez(CACHE / 'mlp.npz', oof=r_mlp['oof_proba'], test=r_mlp['test_proba'])
print(f"MLP CV {r_mlp['cv_mean']:.4f}  ({time.time()-t:.0f}s)")

[fold 0] Macro F1 = 0.3499
[fold 1] Macro F1 = 0.3414
[fold 2] Macro F1 = 0.3744
[fold 3] Macro F1 = 0.3382
[fold 4] Macro F1 = 0.3591
CV Macro F1 = 0.3526 ± 0.0131 (OOF 0.3550)
MLP CV 0.3526  (36s)


## 2) 변이 유전자 트랜스포머
- 환자마다 변이된 유전자만 토큰으로 (+ CLS 토큰), 어텐션 2층, 차원 96
- 변이 128개 초과 환자: 학습 때는 무작위 128개, 예측 때는 무작위 선택 4번 평균 (시드 고정)
- 변이 0개 환자는 CLS 토큰 + stats만으로 예측

In [3]:
t = time.time()
make_tf = lambda: GeneSetTransformer(N_ANY, n_classes=N_CLS, seed=SEED)
r_tf = run_cv(make_tf, X, y, X_test, folds=FOLDS)
np.savez(CACHE / 'genetf.npz', oof=r_tf['oof_proba'], test=r_tf['test_proba'])
print(f"Transformer CV {r_tf['cv_mean']:.4f}  ({time.time()-t:.0f}s)")

[fold 0] Macro F1 = 0.3352
[fold 1] Macro F1 = 0.3322
[fold 2] Macro F1 = 0.3191
[fold 3] Macro F1 = 0.3435
[fold 4] Macro F1 = 0.3489
CV Macro F1 = 0.3358 ± 0.0102 (OOF 0.3373)
Transformer CV 0.3358  (65s)


## 3) 16 최종 섞기에 더하기 (교차 적합)
- `BASE_OOF`, `BASE_TEST`: 16번 6개 버전의 최종 OOF·test 확률
  - 16 노트북에서 `np.savez(CACHE / 'blend6_final.npz', oof=..., test=...)` 한 줄로 저장해 두고 불러옴
- fold k의 딥러닝 비율 w는 **나머지 4개 fold OOF로만** 고르고, fold k에서 채점 (16과 같은 방식)
- fold는 16과 같은 `get_folds(y)`를 그대로 사용

In [4]:
N_FOLDS = len(FOLDS)
folds = np.zeros(len(y), dtype=int)
for k, (_, va) in enumerate(FOLDS):
    folds[va] = k

base = np.load('../submissions/cache16/blend6_final.npz')   # ← 16에서 저장한 파일 이름에 맞게
BASE_OOF, BASE_TEST = base['oof'], base['test']

f1 = lambda yy, p: macro_f1(yy, p.argmax(1))
GRID = [0, 0.1, 0.2, 0.3, 0.4, 0.5]

def crossfit_add(dl_oof, name):
    rows, ws = [], []
    for k in range(N_FOLDS):
        tr, va = folds != k, folds == k
        w = max(GRID, key=lambda w: f1(y[tr], (1 - w) * BASE_OOF[tr] + w * dl_oof[tr]))
        ws.append(w)
        rows.append(dict(fold=k, w=w, base=f1(y[va], BASE_OOF[va]),
                         blend=f1(y[va], (1 - w) * BASE_OOF[va] + w * dl_oof[va])))
    df = pd.DataFrame(rows); df['diff'] = df.blend - df.base
    print(f'[{name}] base {df.base.mean():.4f} → +DL {df.blend.mean():.4f} '
          f'(diff {df["diff"].mean():+.4f}, 오른 fold {(df["diff"] > 0).sum()}/{N_FOLDS}, w {ws})')
    return df, float(np.median(ws))

d_mlp, w_mlp = crossfit_add(r_mlp['oof_proba'], 'MLP')
d_tf,  w_tf  = crossfit_add(r_tf['oof_proba'],  'Transformer')
d_both, w_both = crossfit_add(0.5 * r_mlp['oof_proba'] + 0.5 * r_tf['oof_proba'], 'MLP+TF 반반')
display(pd.concat({'MLP': d_mlp, 'TF': d_tf, 'both': d_both}, axis=1).round(4))

[MLP] base 0.4261 → +DL 0.4273 (diff +0.0012, 오른 fold 3/5, w [0.1, 0.1, 0.1, 0.1, 0.1])
[Transformer] base 0.4261 → +DL 0.4261 (diff +0.0000, 오른 fold 0/5, w [0, 0, 0, 0, 0])
[MLP+TF 반반] base 0.4261 → +DL 0.4238 (diff -0.0023, 오른 fold 0/5, w [0.1, 0.1, 0.1, 0, 0])


MLP                                TF                         both       \
  fold    w    base   blend    diff fold  w    base   blend diff fold    w   
0    0  0.1  0.4262  0.4284  0.0022    0  0  0.4262  0.4262  0.0    0  0.1   
1    1  0.1  0.4255  0.4244 -0.0011    1  0  0.4255  0.4255  0.0    1  0.1   
2    2  0.1  0.4278  0.4261 -0.0017    2  0  0.4278  0.4278  0.0    2  0.1   
3    3  0.1  0.4128  0.4138  0.0010    3  0  0.4128  0.4128  0.0    3  0.0   
4    4  0.1  0.4383  0.4441  0.0058    4  0  0.4383  0.4383  0.0    4  0.0   

                           
     base   blend    diff  
0  0.4262  0.4260 -0.0002  
1  0.4255  0.4254 -0.0000  
2  0.4278  0.4164 -0.0113  
3  0.4128  0.4128  0.0000  
4  0.4383  0.4383  0.0000

## 4) 예측 분포 확인 (제출 전 필수)
- 딥러닝 test 예측이 특정 암종(PRAD, THYM, COAD, SKCM)으로 쏠리는지 확인
- train 비율 대비 배율이 크게 튀는 암종이 있으면 섞기 비율을 낮추거나 제출 보류

In [5]:
def dist(p): return pd.Series(le.inverse_transform(p.argmax(1))).value_counts(normalize=True)
tab = pd.DataFrame({'train %': train.SUBCLASS.value_counts(normalize=True),
                    'base %': dist(BASE_TEST), 'MLP %': dist(r_mlp['test_proba']),
                    'TF %': dist(r_tf['test_proba'])}).fillna(0).mul(100).round(1)
tab['TF/train'] = (tab['TF %'] / tab['train %']).round(2)
tab.sort_values('base %', ascending=False)

,train %,base %,MLP %,TF %,TF/train
COAD,3.6,22.5,2.9,7.5,2.08
STES,6.1,10.5,8.3,2.9,0.48
KIPAN,8.3,6.1,6.4,2.6,0.31
OV,4.1,5.9,4.3,3.9,0.95
BRCA,12.7,5.3,7.6,4.5,0.35
PCPG,2.4,3.8,3.2,2.6,1.08
LGG,3.7,3.8,3.7,3.6,0.97
TGCT,2.0,3.8,2.2,3.2,1.60
THYM,1.6,3.6,2.4,2.9,1.81
SKCM,4.5,3.6,27.8,28.6,6.36


## 5) 제출 파일 저장 (채택 기준 통과한 것만)
- 비율은 교차 적합에서 fold별로 고른 w의 중앙값 사용
- 저장만 하고, 제출은 팀장이 단톡 공유 후 진행

In [6]:
pick = {'MLP': (d_mlp, w_mlp, r_mlp['test_proba']),
        'TF': (d_tf, w_tf, r_tf['test_proba']),
        'both': (d_both, w_both, 0.5 * r_mlp['test_proba'] + 0.5 * r_tf['test_proba'])}
for name, (d, w, dl_test) in pick.items():
    if (d['diff'] > 0).sum() >= 4 and w > 0:
        p = (1 - w) * BASE_TEST + w * dl_test
        save_submission(submission, p, le, name=f'blend6_plus_{name}_w{w}', cv_score=d.blend.mean())
        print('저장:', name, 'w', w)
    else:
        print('기준 미달 → 저장 안 함:', name)

기준 미달 → 저장 안 함: MLP
기준 미달 → 저장 안 함: TF
기준 미달 → 저장 안 함: both


## 결과
| 모델 | 단독 CV | 16 섞기 + DL (교차 적합) | 오른 fold | w | 소요 시간 (4060 Ti) |
| :-- | :-: | :-: | :-: | :-: | :-: |
| 16 6개 버전 (기준) | - | 0.4261 | - | - | - |
| + MLP | 0.3526 ± 0.0131 | 0.4273 (+0.0012) | 3/5 | 0.1 | 36s |
| + Transformer | 0.3358 ± 0.0102 | 0.4261 (+0.0000) | 0/5 | 0 | 65s |
| + MLP·TF 반반 | - | 0.4238 (−0.0023) | 0/5 | 0.1 | - |

**test 예측 분포 (SKCM)**
- train 4.5% / 기준 섞기 3.6% / MLP 27.8% / Transformer 28.6%

## 결론
- **H18a 기각**: MLP는 섞기 OOF를 +0.0012 올렸지만 3/5 fold로 기준(4/5) 미달
- **H18b 기각**: 트랜스포머는 교차 적합에서 5개 fold 모두 비율 0 → 섞기 재료로 쓸모 없음
- 제출 파일 없음, 딥러닝 실험 종료
- **발견: 딥러닝은 test에서 SKCM으로 크게 쏠림** (train 대비 약 6배)
  - SKCM은 변이 수가 가장 많은 암종 → test 환자당 변이 수가 train의 약 2배인 차이를 그대로 받아들인 것으로 추정
  - 학습 fold 빈도 기반 가짜 변이 증강으로도 막지 못함
  - 기준 섞기(트리 + 선형)는 쏠리지 않음 → 이 데이터에서는 트리·선형 섞기가 test 분포 차이에 더 강함
- 교훈 (재확인): CV 숫자만으로 판단하지 말고, 제출 전 test 예측 분포를 꼭 확인